In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/sales/sales.csv",
    header=True,
    inferSchema=True
)
df.show()

+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|user_id|username|               email|age|subscription|transaction_date|region|  city|product_category|  price|sale_amount|   status|store_id|      raw_timestamp|
+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|   1001|user1001|user1001@example.com| 52|       Basic|      2025-02-07|  West|Mumbai|     Electronics| 2418.9|     7256.7|Delivered|    S108|2025-02-07 08:03:00|
|   1002|user1002|user1002@example.com| 30|     Premium|      2025-06-18|  West|Mumbai| Office Supplies|1925.49|    3850.98|  Pending|    S107|2025-06-18 06:25:00|
|   1003|user1003|user1003@example.com| 18|     Premium|      2025-04-29|  West|Mumbai|       Furniture|1046.49|    6278.94|  Pending|    S110|2025-04-29 05:12:00|
|   1004|user100

In [0]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [0]:
# Remove duplicate rows based on user_id and transaction_date
before_count = df.count()
print("Rows before transformation:", before_count)
df_no_duplicates = df.dropDuplicates(["user_id","transaction_date"])
after_count = df_no_duplicates.count()
print("Rows after transformation:", after_count)
print("Duplicate rows removed:", before_count - after_count)
df_no_duplicates.show()

Rows before transformation: 1000
Rows after transformation: 900
Duplicate rows removed: 100
+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|user_id|username|               email|age|subscription|transaction_date|region|  city|product_category|  price|sale_amount|   status|store_id|      raw_timestamp|
+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|   1003|user1003|user1003@example.com| 18|     Premium|      2025-04-29|  West|Mumbai|       Furniture|1046.49|    6278.94|  Pending|    S110|2025-04-29 05:12:00|
|   1030|user1030|user1030@example.com| 58|     Premium|      2025-03-10|  West|Mumbai|        Clothing|4206.49|    4206.49|Cancelled|    S109|2025-03-10 08:47:00|
|   1038|user1038|user1038@example.com| 43|       Basic|      2025-04-23|  West|Mumbai| 

In [0]:
# Filter rows where region is West
# Group by product_category
# Calculate average sale_amount
df_no_duplicates.filter(col("region")=="West") \
.groupBy("product_category") \
.avg("sale_amount") \
.show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|       Furniture|  9871.18752688172|
|        Clothing|11955.315274725273|
|       Groceries|10764.975263157898|
|     Electronics|10983.906235294115|
| Office Supplies|12448.000729166668|
+----------------+------------------+



In [0]:
# Group records by city
# Count the number of records
# Display only cities with count greater than 100
df_no_duplicates.groupBy("city") \
.count() \
.filter(col("count")>100) \
.show()

+---------+-----+
|     city|count|
+---------+-----+
|   Mumbai|  180|
|    Delhi|  170|
|Bengaluru|  160|
|     Pune|  150|
|Ahmedabad|  130|
+---------+-----+



In [0]:
# Filter users whose age is between 18 and 30
# and whose subscription is Premium
df_no_duplicates.filter(
(col("age").between(18,30)) &
(col("subscription")=="Premium")
).show()

+-------+--------+--------------------+---+------------+----------------+------+---------+----------------+-------+-----------+---------+--------+-------------------+
|user_id|username|               email|age|subscription|transaction_date|region|     city|product_category|  price|sale_amount|   status|store_id|      raw_timestamp|
+-------+--------+--------------------+---+------------+----------------+------+---------+----------------+-------+-----------+---------+--------+-------------------+
|   1003|user1003|user1003@example.com| 18|     Premium|      2025-04-29|  West|   Mumbai|       Furniture|1046.49|    6278.94|  Pending|    S110|2025-04-29 05:12:00|
|   1071|user1071|user1071@example.com| 21|     Premium|      2025-05-02|  West|   Mumbai|     Electronics| 4574.7|     9149.4|  Pending|    S101|2025-05-02 19:28:00|
|   1139|user1139|user1139@example.com| 19|     Premium|      2025-02-26|  West|   Mumbai|        Clothing|4253.63|   17014.52|  Pending|    S102|2025-02-26 15:09:00

In [0]:
df_no_duplicates.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [0]:
# Convert raw_timestamp to TimestampType
# Rename the column as event_time
df2 = df_no_duplicates.withColumn(
"event_time",
col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

df2.printSchema()
df2.show()

root
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|user_id|username|               email|age|subscription|transaction_date|region|  city|product_category|  price|sale_amount|   status|store_id|         event_time|
+-------+--------+--------------------+---+------------+----------------+------+------+-----------

In [0]:
# Remove rows where email is null
# or username is an empty string
clean_df = df_no_duplicates.filter(
col("email").isNotNull() &
(col("username")!="")
)
clean_df.show()

+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|user_id|username|               email|age|subscription|transaction_date|region|  city|product_category|  price|sale_amount|   status|store_id|      raw_timestamp|
+-------+--------+--------------------+---+------------+----------------+------+------+----------------+-------+-----------+---------+--------+-------------------+
|   1003|user1003|user1003@example.com| 18|     Premium|      2025-04-29|  West|Mumbai|       Furniture|1046.49|    6278.94|  Pending|    S110|2025-04-29 05:12:00|
|   1030|user1030|user1030@example.com| 58|     Premium|      2025-03-10|  West|Mumbai|        Clothing|4206.49|    4206.49|Cancelled|    S109|2025-03-10 08:47:00|
|   1038|user1038|user1038@example.com| 43|       Basic|      2025-04-23|  West|Mumbai|       Groceries|3607.47|   10822.41|     NULL|    S107|2025-04-23 06:42:00|
|   1044|user104

In [0]:
# Calculate minimum, maximum, and average price
clean_df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|       104.29|      5999.05|3143.1558579881657|
+-------------+-------------+------------------+



In [0]:
# Remove duplicate rows
# Fill null values in price with 0
# Group data by store_id
# Calculate total revenue for each store
final_df = (
    df
    .dropDuplicates()
    .na.fill({"price":0})
    .groupBy("store_id")
    .agg(sum("price").alias("Total Revenue"))
)

# Display the final result
final_df.show()

+--------+------------------+
|store_id|     Total Revenue|
+--------+------------------+
|    S107|         299966.38|
|    S102|366897.69000000006|
|    S105| 319005.2700000001|
|    S109|         286154.46|
|    S104| 303184.8299999999|
|    S108| 323899.6700000001|
|    S103| 263528.4599999999|
|    S110| 304063.5199999999|
|    S101| 316126.6000000001|
|    S106|282822.39999999997|
+--------+------------------+

